# Préparer le fichier parquet des images

**À lancer une seule fois, par une seule personne.** Le reste de l'équipe utilise ensuite le fichier partagé sur Drive.

Ce notebook télécharge environ 10 Go de patchs depuis Hugging Face, garde les 39 000 patchs du sous-échantillon, vérifie le résultat et le copie dans `Mon Drive/SPE1`.

- Pas besoin de GPU : le processeur suffit, car on ne fait que trier des fichiers.
- Compter 10 à 20 minutes, et laisser l'onglet ouvert.
- Dans le menu Exécution, choisir Tout exécuter, puis autoriser l'accès à Google Drive.

La fonction de reconstruction vient du notebook de démarrage du cours, sans changement de logique.

## 1. Récupérer les fichiers du dépôt

Le dépôt contient la liste des patchs à garder et le découpage de l'équipe.

In [ ]:
import os, sys, json, time, shutil, urllib.request
import numpy as np, pandas as pd

if not os.path.exists("depot"):
    !git clone --depth 1 https://github.com/kenzo-kouokam/detection-metastases-camelyon17.git depot
sys.path.insert(0, "depot/p1_data")
from charger_donnees import charger_donnees

IDS_CSV = "depot/donnees/ids_sous_echantillon.csv"
FICHIER = "camelyon17_sous_echantillon.parquet"
DOSSIER_DRIVE = "/content/drive/MyDrive/SPE1"
CIBLE = f"{DOSSIER_DRIVE}/{FICHIER}"

## 2. Connecter Google Drive

On le fait dès le début : si l'autorisation échoue, on le sait avant d'avoir attendu 20 minutes.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DOSSIER_DRIVE, exist_ok=True)
print("Le fichier existe déjà sur Drive :", os.path.exists(CIBLE))

## 3. Fabriquer le fichier

Les fichiers sont téléchargés un par un, et chacun est effacé dès qu'on en a extrait les patchs utiles : cela évite de saturer le disque.

In [ ]:
def reconstruire_depuis_hf(ids_csv, sortie):
    import pyarrow.parquet as pq, pyarrow as pa
    ids = pd.read_csv(ids_csv)
    voulu = dict(zip(ids.image_id, ids.split))
    api = "https://huggingface.co/api/datasets/wltjr1007/Camelyon17-WILDS/parquet/default"
    shards = json.load(urllib.request.urlopen(api))
    morceaux = []
    for split, urls in shards.items():
        for i, url in enumerate(urls):
            t0 = time.time(); f = f"tmp_{split}_{i}.parquet"
            urllib.request.urlretrieve(url, f)
            t = pq.read_table(f)
            ids_shard = t.column("image_id").to_numpy()
            garder = np.flatnonzero(np.isin(ids_shard, list(voulu)))
            if len(garder):
                s = t.take(pa.array(garder)).to_pandas()
                s["image"] = s["image"].apply(lambda d: d["bytes"])
                morceaux.append(s)
            os.remove(f)
            print(f"{split} {i + 1}/{len(urls)} : {len(garder)} patchs gardés ({time.time() - t0:.0f} s)")
    df = pd.concat(morceaux, ignore_index=True)
    df["split"] = df.image_id.map(voulu)
    df.to_parquet(sortie, index=False)

if os.path.exists(CIBLE):
    print("Rien à fabriquer : le fichier est déjà sur Drive.")
    FICHIER = CIBLE
elif not os.path.exists(FICHIER):
    reconstruire_depuis_hf(IDS_CSV, FICHIER)

## 4. Vérifier le résultat

Si quelque chose ne va pas, cette cellule s'arrête sur une erreur et rien n'est copié. Elle contrôle : 39 000 patchs, aucun doublon, la bonne liste, des images de 96 x 96 pixels, aucune fuite entre les jeux.

In [ ]:
df = charger_donnees(FICHIER, "depot/p1_data/sorties/splits_par_patch.csv")
print(len(df), "patchs : vérifications réussies")
print(f"Taille du fichier : {os.path.getsize(FICHIER) / 1e9:.2f} Go")
pd.crosstab(df.split_equipe, [df.center, df.label])

## 5. Copier sur Google Drive

In [ ]:
if FICHIER != CIBLE:
    shutil.copy(FICHIER, CIBLE)
    drive.flush_and_unmount()
    drive.mount("/content/drive")
assert os.path.getsize(CIBLE) == os.path.getsize(FICHIER), "Copie incomplète : relancer cette cellule."
print("Terminé :", CIBLE)

## 6. Partager avec l'équipe

1. Ouvrir https://drive.google.com, puis Mon Drive.
2. Clic droit sur le dossier `SPE1`, Partager, ajouter les adresses de l'équipe avec le rôle **Lecteur**.
3. Copier le lien du dossier et l'envoyer à l'équipe.

Le rôle Lecteur empêche une suppression ou une modification par erreur : tout le monde travaille ainsi sur le même fichier.